In [ ]:
pip install pandas


In [2]:
# Import the necessary libraries
import pandas as pd
import numpy as np

# Mount your Google Drive to give Colab access to your files
from google.colab import drive
drive.mount('/content/drive')

# --- IMPORTANT: Define the path to your MIMIC-IV CSV files in Google Drive ---

DRIVE_PATH = '/content/drive/MyDrive/mimic_iv_data/'

print("Environment setup complete. Google Drive is mounted.")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
print("\n--- Starting Part 1: Discovery Phase ---")

# --- Load the dictionary files ---
try:
    d_labitems_df = pd.read_csv(DRIVE_PATH + 'd_labitems.csv')
    d_icd_diagnoses_df = pd.read_csv(DRIVE_PATH + 'd_icd_diagnoses.csv')
    print("Successfully loaded d_labitems.csv and d_icd_diagnoses.csv.")
except FileNotFoundError:
    print("ERROR: Could not find the dictionary files. Please check your DRIVE_PATH.")
    # Stop execution if files are not found
    exit()

# --- 1a. Discover ICD-10 Codes for Sepsis and AKI ---
print("\n--- Discovering ICD-10 Codes ---")

# Search for relevant Sepsis codes (ICD-10 only)
sepsis_codes_df = d_icd_diagnoses_df[
    d_icd_diagnoses_df['long_title'].str.contains('Sepsis', case=False, na=False) &
    (d_icd_diagnoses_df['icd_version'] == 10)
]
print("\nPotential ICD-10 Codes for Sepsis:")
print(sepsis_codes_df)

# Search for relevant AKI codes (ICD-10 only)
aki_codes_df = d_icd_diagnoses_df[
    d_icd_diagnoses_df['long_title'].str.contains('Acute kidney', case=False, na=False) &
    (d_icd_diagnoses_df['icd_version'] == 10)
]
print("\nPotential ICD-10 Codes for Acute Kidney Injury:")
print(aki_codes_df)


# --- 1b. Discover Item IDs for Lab Panels ---
print("\n--- Discovering Lab Test Item IDs ---")

# Define the names of the tests from the base paper's appendix
cbc_test_names = ['Hematocrit', 'Platelet Count', 'White Blood Cell Count', 'Red Blood Cell Count', 'Hemoglobin']
cmp_test_names = ['Creatinine', 'Bicarbonate', 'Glucose', 'Potassium', 'Urea Nitrogen', 'Anion Gap', 'Lactate']

# Filter the d_labitems dataframe to find the corresponding itemids
cbc_items_df = d_labitems_df[d_labitems_df['label'].isin(cbc_test_names)]
cmp_items_df = d_labitems_df[d_labitems_df['label'].isin(cmp_test_names)]

print("\nDiscovered CBC Panel Item IDs:")
print(cbc_items_df)
print("\nDiscovered CMP Panel Item IDs:")
print(cmp_items_df)




--- Starting Part 1: Discovery Phase ---
Successfully loaded d_labitems.csv and d_icd_diagnoses.csv.

--- Discovering ICD-10 Codes ---

Potential ICD-10 Codes for Sepsis:
      icd_code  icd_version                                         long_title
12257     A021           10                                  Salmonella sepsis
12413     A227           10                                     Anthrax sepsis
12435     A267           10                              Erysipelothrix sepsis
12470     A327           10                                   Listerial sepsis
12529      A40           10                               Streptococcal sepsis
12530     A400           10               Sepsis due to streptococcus, group A
12531     A401           10               Sepsis due to streptococcus, group B
12532     A403           10             Sepsis due to Streptococcus pneumoniae
12533     A408           10                         Other streptococcal sepsis
12534     A409           10           

In [ ]:
import pandas as pd
import numpy as np



print("\n--- Starting Part 1: Discovery Phase (Comprehensive) ---")

# --- Load the dictionary files ---
try:
    d_labitems_df = pd.read_csv(DRIVE_PATH + 'd_labitems.csv')
    d_icd_diagnoses_df = pd.read_csv(DRIVE_PATH + 'd_icd_diagnoses.csv')
    print(" Successfully loaded d_labitems.csv and d_icd_diagnoses.csv.")
except FileNotFoundError as e:
    print(f" ERROR: Could not find dictionary files. Please check your DRIVE_PATH. Details: {e}")
    exit()

# --- PRELIMINARY STEP: Confirm Column Names ---
print("\n--- Confirming column names from dictionary files ---")
print(f"Columns in d_icd_diagnoses.csv: {d_icd_diagnoses_df.columns.tolist()}")
print(f"Columns in d_labitems.csv: {d_labitems_df.columns.tolist()}")
# This confirms we can use 'long_title', 'icd_version', 'label', 'fluid', and 'itemid'.

# --- 1a. Discover ICD-10 Codes ---
# This code is now confirmed to work with the correct column names.
sepsis_codes_df = d_icd_diagnoses_df[d_icd_diagnoses_df['long_title'].str.contains('Sepsis', case=False, na=False) & (d_icd_diagnoses_df['icd_version'] == 10)]
aki_codes_df = d_icd_diagnoses_df[d_icd_diagnoses_df['long_title'].str.contains('Acute kidney', case=False, na=False) & (d_icd_diagnoses_df['icd_version'] == 10)]

# --- 1b. Discover Item IDs for ALL Lab Panels ---
cbc_test_names = ['Hematocrit', 'Platelet Count', 'White Blood Cell Count', 'Red Blood Cell Count', 'Hemoglobin']
cmp_test_names = ['Creatinine', 'Bicarbonate', 'Glucose', 'Potassium', 'Urea Nitrogen', 'Anion Gap', 'Lactate', 'Aspartate Aminotransferase', 'Bilirubin', 'Chloride', 'Albumin', 'Sodium']
abg_test_names = ['Base Excess', 'pH', 'Oxygen Saturation', 'Inspired O2 Fraction', 'Arterial Pressure']
aptt_test_names = ['PTT', 'INR', 'Prothrombin Time']

def find_item_ids(df, test_names):
    # Using a more precise regex with word boundaries (\b) to avoid partial matches
    regex_pattern = r'\b(' + '|'.join(test_names) + r')\b'
    return df[df['label'].str.contains(regex_pattern, case=False, na=False) & (df['fluid'] == 'Blood')]['itemid'].unique().tolist()

DISCOVERED_CBC_ITEMIDS = find_item_ids(d_labitems_df, cbc_test_names)
DISCOVERED_CMP_ITEMIDS = find_item_ids(d_labitems_df, cmp_test_names)
DISCOVERED_ABG_ITEMIDS = find_item_ids(d_labitems_df, abg_test_names)
DISCOVERED_APTT_ITEMIDS = find_item_ids(d_labitems_df, aptt_test_names)

# These lists should be populated based on the output from the discovery above for maximum accuracy,
# but using a pre-vetted list is also a valid scientific approach.
DISCOVERED_SEPSIS_CODES = ['A419', 'R6520', 'R6521']
DISCOVERED_AKI_CODES = ['N170', 'N171', 'N172', 'N179']

print("\n---  Discovered Item IDs for All Panels ---")
print(f"CBC Item IDs ({len(DISCOVERED_CBC_ITEMIDS)} found): {DISCOVERED_CBC_ITEMIDS}")
print(f"CMP Item IDs ({len(DISCOVERED_CMP_ITEMIDS)} found): {DISCOVERED_CMP_ITEMIDS}")
print(f"ABG Item IDs ({len(DISCOVERED_ABG_ITEMIDS)} found): {DISCOVERED_ABG_ITEMIDS}")
print(f"APTT Item IDs ({len(DISCOVERED_APTT_ITEMIDS)} found): {DISCOVERED_APTT_ITEMIDS}")
print("\n---  Part 1: Discovery Phase Complete ---")


--- Starting Part 1: Discovery Phase (Comprehensive) ---
✅ Successfully loaded d_labitems.csv and d_icd_diagnoses.csv.

--- Confirming column names from dictionary files ---
Columns in d_icd_diagnoses.csv: ['icd_code', 'icd_version', 'long_title']
Columns in d_labitems.csv: ['itemid', 'label', 'fluid', 'category']

--- ✅ Discovered Item IDs for All Panels ---
CBC Item IDs (29 found): [50810, 50811, 50852, 50855, 51212, 51221, 51222, 51223, 51224, 51225, 51265, 51285, 51631, 51638, 51639, 51640, 51641, 51642, 51643, 51644, 51645, 51646, 51647, 52028, 52032, 52128, 52129, 52157, 53189]
CMP Item IDs (40 found): [50803, 50806, 50809, 50813, 50822, 50824, 50862, 50868, 50882, 50883, 50884, 50885, 50902, 50912, 50931, 50954, 50971, 50983, 51006, 51568, 51569, 51570, 52022, 52024, 52027, 52434, 52442, 52452, 52455, 52500, 52535, 52546, 52569, 52610, 52623, 52647, 53085, 53089, 53138, 53154]
ABG Item IDs (3 found): [50802, 50817, 50820]
APTT Item IDs (7 found): [51237, 51275, 51675, 52165, 52

/tmp/ipython-input-4-1414520979.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return df[df['label'].str.contains(regex_pattern, case=False, na=False) & (df['fluid'] == 'Blood')]['itemid'].unique().tolist()


In [ ]:

# ==============================================================================
# PART 2: MAIN ANALYSIS USING ALL DISCOVERED CODES
# ==============================================================================

print("\n--- Starting Part 2: Analysis Phase (Complete) ---")


DISCOVERED_SEPSIS_CODES = ['A021', 'A227', 'A267', 'A327', 'A400', 'A401', 'A403', 'A408', 'A409', 'A4101', 'A4102', 'A411', 'A412', 'A413', 'A414', 'A4150', 'A4151', 'A4152', 'A4153', 'A4159', 'A4181', 'A4189', 'A419', 'A427', 'A5486', 'B377', 'O0337', 'O0387', 'O0487', 'O0737', 'O0882', 'O85', 'O8604', 'P360', 'P3610', 'P3619', 'P362', 'P3630', 'P3639', 'P364', 'P365', 'P368', 'P369', 'R6520', 'R6521', 'T8144', 'T8144XA', 'T8144XD', 'T8144XS']
DISCOVERED_AKI_CODES = ['N170', 'N171', 'N172', 'N178', 'N179', 'O904']

DISCOVERED_CBC_ITEMIDS = [50811, 51221, 51222, 51265, 51638, 51639, 51640, 52028, 53189]
DISCOVERED_CMP_ITEMIDS = [50809, 50813, 50868, 50882, 50912, 50931, 50971, 51006, 52442, 52500, 52546, 52569, 52610, 52647, 53154]
DISCOVERED_ABG_ITEMIDS = [50802, 50817, 50820]
DISCOVERED_APTT_ITEMIDS = [51237, 51275, 51675, 52165, 52166, 52167, 52923]


print("Using the following codes and IDs for analysis:")
print(f"Sepsis Codes ({len(DISCOVERED_SEPSIS_CODES)}): {DISCOVERED_SEPSIS_CODES}")
print(f"AKI Codes ({len(DISCOVERED_AKI_CODES)}): {DISCOVERED_AKI_CODES}")
print(f"CBC Item IDs ({len(DISCOVERED_CBC_ITEMIDS)}): {DISCOVERED_CBC_ITEMIDS}")
print(f"CMP Item IDs ({len(DISCOVERED_CMP_ITEMIDS)}): {DISCOVERED_CMP_ITEMIDS}")
print(f"ABG Item IDs ({len(DISCOVERED_ABG_ITEMIDS)}): {DISCOVERED_ABG_ITEMIDS}")
print(f"APTT Item IDs ({len(DISCOVERED_APTT_ITEMIDS)}): {DISCOVERED_APTT_ITEMIDS}")

# --- Build Patient Cohorts ---
print("\n--- Building Patient Cohorts ---")
try:
    diagnoses_df = pd.read_csv(DRIVE_PATH + 'diagnoses_icd.csv', usecols=['subject_id', 'icd_code', 'icd_version'])
    diagnoses_df = diagnoses_df[diagnoses_df['icd_version'] == 10]

    sepsis_patients_df = diagnoses_df[diagnoses_df['icd_code'].isin(DISCOVERED_SEPSIS_CODES)]
    sepsis_subject_ids = sepsis_patients_df['subject_id'].unique()
    print(f"Successfully built Sepsis cohort with {len(sepsis_subject_ids)} unique patients.")

    aki_patients_df = diagnoses_df[diagnoses_df['icd_code'].isin(DISCOVERED_AKI_CODES)]
    aki_subject_ids = aki_patients_df['subject_id'].unique()
    print(f"Successfully built AKI cohort with {len(aki_subject_ids)} unique patients.")

except FileNotFoundError as e:
    print(f" ERROR: Could not find diagnoses_icd.csv. Details: {e}")
    exit()

# --- The robust analysis function ---
def analyze_cohort_missingness(cohort_name, subject_ids, panel_name, itemids, labevents_filepath):
    print(f"\n--- Starting Missingness Analysis for {cohort_name} Cohort and {panel_name} Panel ---")
    if len(subject_ids) == 0:
        print("Cohort contains no patients. Skipping analysis.")
        return

    relevant_labs_chunks = []
    chunk_size = 1000000
    try:
        labevents_iterator = pd.read_csv(labevents_filepath, chunksize=chunk_size, low_memory=False, usecols=['subject_id', 'itemid', 'valuenum'])
        print(f"Processing {labevents_filepath}... (This may take several minutes)")
        for i, chunk in enumerate(labevents_iterator):
            filtered_chunk = chunk[chunk['subject_id'].isin(subject_ids) & chunk['itemid'].isin(itemids)]
            if not filtered_chunk.empty:
                relevant_labs_chunks.append(filtered_chunk)
            print(f"  Processed chunk {i+1}...")
    except FileNotFoundError as e:
        print(f" ERROR: Could not find {labevents_filepath}. Details: {e}")
        return

    if not relevant_labs_chunks:
        print(f"Result: No lab events found for the {panel_name} panel in the {cohort_name} cohort.")
        return

    relevant_labs_df = pd.concat(relevant_labs_chunks)

    # Calculate Panel-Level Missingness
    patients_with_labs = relevant_labs_df['subject_id'].unique()
    panel_level_missingness = 1 - (len(patients_with_labs) / len(subject_ids))

    # Calculate Sporadic Missingness
    if len(relevant_labs_df) > 0:
        sporadic_missingness = relevant_labs_df['valuenum'].isnull().sum() / len(relevant_labs_df)
    else:
        sporadic_missingness = 0

    print(f"\n--- Results for {cohort_name} Cohort, {panel_name} Panel ---")
    print(f"Total patients in cohort: {len(subject_ids)}")
    print(f"Patients with at least one panel test result: {len(patients_with_labs)}")
    print(f"==> Panel-Level Missingness: {panel_level_missingness:.2%}")
    print(f"\nTotal measurements found for this panel: {len(relevant_labs_df)}")
    print(f"Number of null/missing measurement values: {relevant_labs_df['valuenum'].isnull().sum()}")
    print(f"==> Sporadic Missingness: {sporadic_missingness:.2%}")


# --- Run the FULL analysis for both cohorts and ALL FOUR panels ---
labevents_filepath = DRIVE_PATH + 'labevents.csv'

# Sepsis Analysis
print("\n" + "="*50 + "\nPERFORMING SEPSIS COHORT ANALYSIS\n" + "="*50)
analyze_cohort_missingness("Sepsis", sepsis_subject_ids, "CBC", DISCOVERED_CBC_ITEMIDS, labevents_filepath)
analyze_cohort_missingness("Sepsis", sepsis_subject_ids, "CMP", DISCOVERED_CMP_ITEMIDS, labevents_filepath)
analyze_cohort_missingness("Sepsis", sepsis_subject_ids, "ABG", DISCOVERED_ABG_ITEMIDS, labevents_filepath)
analyze_cohort_missingness("Sepsis", sepsis_subject_ids, "APTT", DISCOVERED_APTT_ITEMIDS, labevents_filepath)

# AKI Analysis
print("\n" + "="*50 + "\nPERFORMING AKI COHORT ANALYSIS\n" + "="*50)
analyze_cohort_missingness("AKI", aki_subject_ids, "CBC", DISCOVERED_CBC_ITEMIDS, labevents_filepath)
analyze_cohort_missingness("AKI", aki_subject_ids, "CMP", DISCOVERED_CMP_ITEMIDS, labevents_filepath)
analyze_cohort_missingness("AKI", aki_subject_ids, "ABG", DISCOVERED_ABG_ITEMIDS, labevents_filepath)
analyze_cohort_missingness("AKI", aki_subject_ids, "APTT", DISCOVERED_APTT_ITEMIDS, labevents_filepath)

print("\n---  Analysis Complete ---")


--- Starting Part 2: Analysis Phase (Complete) ---
Using the following codes and IDs for analysis:
Sepsis Codes (49): ['A021', 'A227', 'A267', 'A327', 'A400', 'A401', 'A403', 'A408', 'A409', 'A4101', 'A4102', 'A411', 'A412', 'A413', 'A414', 'A4150', 'A4151', 'A4152', 'A4153', 'A4159', 'A4181', 'A4189', 'A419', 'A427', 'A5486', 'B377', 'O0337', 'O0387', 'O0487', 'O0737', 'O0882', 'O85', 'O8604', 'P360', 'P3610', 'P3619', 'P362', 'P3630', 'P3639', 'P364', 'P365', 'P368', 'P369', 'R6520', 'R6521', 'T8144', 'T8144XA', 'T8144XD', 'T8144XS']
AKI Codes (6): ['N170', 'N171', 'N172', 'N178', 'N179', 'O904']
CBC Item IDs (9): [50811, 51221, 51222, 51265, 51638, 51639, 51640, 52028, 53189]
CMP Item IDs (15): [50809, 50813, 50868, 50882, 50912, 50931, 50971, 51006, 52442, 52500, 52546, 52569, 52610, 52647, 53154]
ABG Item IDs (3): [50802, 50817, 50820]
APTT Item IDs (7): [51237, 51275, 51675, 52165, 52166, 52167, 52923]

--- Building Patient Cohorts ---
Successfully built Sepsis cohort with 11321

In [1]:
col = 'respiratory_rate'  # change to your column name if needed
x = pd.to_numeric(df[col], errors='coerce')

# 1) Summary + std
x.describe(), float(x.std())

# 2) Inspect extremes
x.sort_values(ascending=False).head(20)

# 3) Count implausible values
(x > 100).sum(), (x > 1000).sum(), x.isin([999, 9999, -1, -9]).sum()

# 4) Distribution view
x.hist(bins=60)

NameError: name 'pd' is not defined

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load sepsis feature matrix and processed split
fm = pd.read_csv("../data/preprocessed/sepsis_feature_matrix.csv")       # raw 0–24h aggregates
X = pd.read_csv("../data/processed/sepsis/train_X.csv")                  # model inputs

# Columns to inspect
lact_cols = [c for c in fm.columns if "lactate" in c]                 # e.g., cmp_lactate_mean/min/max
rr_cols   = [c for c in fm.columns if "respiratory_rate" in c]        # *_mean/min/max
lact_cols, rr_cols

(['cmp_lactate'],
 ['respiratory_rate_mean', 'respiratory_rate_min', 'respiratory_rate_max'])

In [5]:
def quick_check(df, cols):
    print(df[cols].describe())
    print("\nTop extremes across these cols:")
    print(df[cols].stack().sort_values(ascending=False).head(20))

quick_check(fm, lact_cols)
quick_check(fm, rr_cols)

         cmp_lactate
count   11646.000000
mean       39.039956
std      3941.616244
min         0.300000
25%         1.300000
50%         1.833333
75%         2.750000
max    425368.400000

Top extremes across these cols:
4085   cmp_lactate    425368.400000
7588   cmp_lactate        28.000000
6011   cmp_lactate        24.600000
12895  cmp_lactate        24.528571
14996  cmp_lactate        23.333333
4666   cmp_lactate        23.000000
526    cmp_lactate        22.000000
3233   cmp_lactate        22.000000
13975  cmp_lactate        21.500000
6202   cmp_lactate        21.250000
1466   cmp_lactate        21.185714
7776   cmp_lactate        21.000000
412    cmp_lactate        20.750000
1913   cmp_lactate        20.636364
1528   cmp_lactate        20.428571
2239   cmp_lactate        20.333333
12834  cmp_lactate        20.241667
10931  cmp_lactate        20.000000
6123   cmp_lactate        20.000000
15192  cmp_lactate        19.788889
dtype: float64
       respiratory_rate_mean  respiratory_r

In [6]:
# Lactate plausible ~0.2–20 mmol/L (rarely >20)
l = pd.to_numeric(fm[lact_cols[0]], errors='coerce')  # choose mean first
print("Lactate > 20:", int((l > 20).sum()), " > 100:", int((l > 100).sum()))
print(l.value_counts().head(10))  # look for sentinels like 99, 999, -1

# Respiratory rate plausible ~4–80 breaths/min
r = pd.to_numeric(fm[rr_cols[0]], errors='coerce')
print("RR > 80:", int((r > 80).sum()), " > 1000:", int((r > 1000).sum()))
print(r.value_counts().head(10))

Lactate > 20: 17  > 100: 1
cmp_lactate
1.1    410
1.3    380
1.0    362
1.2    359
1.4    349
1.5    346
1.6    329
1.7    299
1.8    293
0.9    286
Name: count, dtype: int64
RR > 80: 3  > 1000: 1
respiratory_rate_mean
17.0    79
18.0    68
19.0    63
21.0    59
23.0    58
20.0    53
16.0    52
22.0    52
18.5    42
19.5    40
Name: count, dtype: int64


In [ ]:
# Offending rows (indices)
fm.loc[[4085], ['cmp_lactate']]  # lactate ~425,368
fm.loc[[11478], ['respiratory_rate_max', 'respiratory_rate_mean']]

In [7]:
# Offending rows (indices)
fm.loc[[4085], ['cmp_lactate']]  # lactate ~425,368
fm.loc[[11478], ['respiratory_rate_max', 'respiratory_rate_mean']]

,respiratory_rate_max,respiratory_rate_mean
11478,7000400.0,269266.769231


In [9]:
import pandas as pd
import numpy as np

# Load processed features (works regardless of where preprocessed FM lives)
X_sepsis = pd.read_csv("../data/processed/sepsis/train_X.csv")
X_aki    = pd.read_csv("../data/processed/aki/train_X.csv")

sentinels = {-1, -9, 99, 999, 9999}

# Physiologic ranges (edit/extend as needed)
bounds = {
    # vitals
    "respiratory_rate": (4, 80),
    "heart_rate": (20, 220),
    "sbp": (50, 280),
    "dbp": (20, 200),
    "spo2|o2_saturation": (0, 100),
    "temperature_c": (30, 43),
    # common labs
    "cmp_lactate|lactate": (0.2, 20),
    "cmp_creatinine|creatinine": (0.1, 15),
    "cmp_bicarbonate|bicarbonate": (5, 50),
    "cmp_glucose|glucose": (20, 1000),
    "cmp_potassium|potassium": (2, 7),
    "cmp_bun|bun": (1, 200),
    "cmp_aniongap|aniongap": (0, 50),
    "cbc_wbc|wbc": (0.1, 200),
    "cbc_hemoglobin|hemoglobin": (3, 25),
    "cbc_hematocrit|hematocrit": (5, 80),
    "cbc_platelet|platelet": (5, 1500),
    "aptt_inr|inr": (0.5, 10),
    "aptt_ptt|ptt|pt": (5, 200),
    "abg_ph|ph": (6.8, 7.8),
    "base_excess": (-30, 30),
    "sodium": (110, 180),
    "chloride": (70, 130),
    "calcium": (4, 16),
    "magnesium": (0.5, 6),
    "phosphate": (0.5, 15),
    "fibrinogen": (50, 1200),
    "bilirubin_total|bilirubin": (0.1, 40),
    "alt|ast|alp": (0, 20000),
}

def col_bounds(colname):
    for k, rng in bounds.items():
        if any(tok in colname for tok in k.split("|")):
            return rng
    return None

def scan_df(df: pd.DataFrame, name: str):
    rows = []
    for c in df.columns:
        s = pd.to_numeric(df[c], errors="coerce")
        if s.notna().sum() == 0:
            continue
        low_high = col_bounds(c)
        # sentinel count
        n_sentinel = int(s.isin(sentinels).sum())
        # absurd magnitude guard
        n_gt_1e4 = int((s.abs() > 1e4).sum())
        # physiologic check (if known)
        n_out_range = None
        if low_high:
            lo, hi = low_high
            n_out_range = int(((s < lo) | (s > hi)).sum())
        # IQR-based outliers (robust fallback)
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lo_iqr = q1 - 1.5 * iqr
        hi_iqr = q3 + 1.5 * iqr
        n_iqr = int(((s < lo_iqr) | (s > hi_iqr)).sum())
        rows.append({
            "feature": c,
            "n": int(s.notna().sum()),
            "std": float(s.std(skipna=True)),
            "max": float(s.max(skipna=True)),
            "n_sentinel": n_sentinel,
            "n_absurd(>1e4)": n_gt_1e4,
            "n_out_of_physio" if low_high else "n_out_of_physio(NA)": n_out_range if low_high else None,
            "n_iqr_outliers": n_iqr,
        })
    out = pd.DataFrame(rows)
    # Rank by any obvious problems first
    if "n_out_of_physio" in out.columns:
        sort_cols = ["n_absurd(>1e4)", "n_sentinel", "n_out_of_physio", "n_iqr_outliers"]
    else:
        sort_cols = ["n_absurd(>1e4)", "n_sentinel", "n_iqr_outliers"]
    out = out.sort_values(sort_cols, ascending=False).reset_index(drop=True)
    print(f"\n=== Outlier scan: {name} ===")
    display(out.head(30))
    return out

scan_sepsis = scan_df(X_sepsis, "Sepsis train_X.csv")
scan_aki    = scan_df(X_aki,    "AKI train_X.csv")


=== Outlier scan: Sepsis train_X.csv ===


,feature,n,std,max,n_sentinel,n_absurd(>1e4),n_out_of_physio(NA),n_iqr_outliers,n_out_of_physio
0,abg_ph,19092,1.225177,3.866601,0,0,NaN,2099,19092.0
1,temperature_c_min,19092,0.880590,2.808733,0,0,NaN,1694,19092.0
2,dbp_min,19092,1.080633,9.947167,0,0,NaN,1149,19092.0
3,sbp_min,19092,1.230549,8.723528,0,0,NaN,1146,19092.0
4,cmp_glucose,19092,1.279975,14.119114,0,0,NaN,1047,19092.0
5,sbp_mean,19092,1.076089,21.746982,0,0,NaN,980,19092.0
6,heart_rate_min,19092,1.253922,4.988803,0,0,NaN,466,19092.0
7,respiratory_rate_max,19092,0.786967,108.729010,0,0,NaN,388,19092.0
8,respiratory_rate_mean,19092,0.786970,108.728843,0,0,NaN,321,19092.0
9,dbp_max,19092,0.789170,107.571427,0,0,NaN,781,19091.0



=== Outlier scan: AKI train_X.csv ===


,feature,n,std,max,n_sentinel,n_absurd(>1e4),n_out_of_physio(NA),n_iqr_outliers,n_out_of_physio
0,ast_max,96438,1022.312426,42606.000,51,274,NaN,44025,37.0
1,ast_mean,96438,755.628708,34532.500,41,129,NaN,44915,14.0
2,ast_min,96438,561.311322,26790.000,56,63,NaN,46267,3.0
3,alt_max,96438,561.324745,24220.000,36,58,NaN,45769,2.0
4,alt_mean,96438,459.048348,20243.334,25,24,NaN,45816,1.0
5,dbp_max,96438,949.488649,114109.000,993,13,NaN,3110,144.0
6,alt_min,96438,376.986258,14770.000,36,13,NaN,45681,0.0
7,sbp_max,96438,3349.272704,1025100.000,170,9,NaN,2316,33.0
8,spo2_max,96438,4989.256581,981023.000,9003,8,NaN,6339,62.0
9,spo2_mean,96438,199.607626,39325.400,400,6,NaN,1818,53.0


In [10]:
import pandas as pd
X_sepsis = pd.read_csv("../data/processed/sepsis/train_X.csv")
X_aki    = pd.read_csv("../data/processed/aki/train_X.csv")

len(X_sepsis.columns), len(X_aki.columns)
# Expect (37, 108)

# If counts differ, see which are present:
list(X_sepsis.columns), list(X_aki.columns)

(['age',
  'gender',
  'heart_rate_mean',
  'sbp_mean',
  'dbp_mean',
  'respiratory_rate_mean',
  'spo2_mean',
  'temperature_c_mean',
  'heart_rate_min',
  'sbp_min',
  'dbp_min',
  'respiratory_rate_min',
  'spo2_min',
  'temperature_c_min',
  'heart_rate_max',
  'sbp_max',
  'dbp_max',
  'respiratory_rate_max',
  'spo2_max',
  'temperature_c_max',
  'abg_base_excess',
  'cmp_lactate',
  'abg_o2_saturation',
  'abg_ph',
  'cmp_aniongap',
  'cmp_bicarbonate',
  'cmp_creatinine',
  'cmp_glucose',
  'cmp_potassium',
  'cmp_bun',
  'cbc_hematocrit',
  'cbc_hemoglobin',
  'aptt_inr',
  'cbc_platelet',
  'aptt_ptt',
  'cbc_rbc',
  'cbc_wbc'],
 ['age',
  'gender',
  'alp_max',
  'alp_mean',
  'alp_min',
  'alt_max',
  'alt_mean',
  'alt_min',
  'aniongap_max',
  'aniongap_mean',
  'aniongap_min',
  'ast_max',
  'ast_mean',
  'ast_min',
  'base_excess_max',
  'base_excess_mean',
  'base_excess_min',
  'bicarbonate_max',
  'bicarbonate_mean',
  'bicarbonate_min',
  'bilirubin_total_max',
  '

In [14]:

len(X_sepsis.columns), len(X_aki.columns)


(37, 104)

In [16]:
# After running the AKI augmentation script
train_cols = pd.read_csv("../data/processed/aki/train_X.csv", nrows=0).columns
raw_cols   = pd.read_csv("../data/preprocessed/aki_feature_matrix.csv", nrows=0).columns
expected   = [c for c in raw_cols if c not in {"stay_id","subject_id","hadm_id","kdigo_aki"}]
set(expected) == set(train_cols)  # should be True

True

In [18]:
import pandas as pd
import numpy as np

# Load model inputs (what you actually train on)
X_sepsis = pd.read_csv("../data/processed/sepsis/train_X.csv")
X_aki    = pd.read_csv("../data/processed/aki/train_X.csv")

# Also load preprocessed matrices (have IDs; helpful to trace offenders)
fm_sepsis = pd.read_csv("../data/preprocessed/sepsis_feature_matrix.csv") if (pd.Series(["../data/preprocessed/sepsis_feature_matrix.csv"]).map(lambda p: __import__('pathlib').Path(p).exists())[0]) else None
fm_aki    = pd.read_csv("../data/preprocessed/aki_feature_matrix.csv")

sentinels = {-1, -9, 99, 999, 9999}

# Physiologic/unit sanity bounds (edit if your clinicians prefer different caps)
bounds = {
    # vitals
    "respiratory_rate": (4, 80),
    "heart_rate": (20, 220),
    "sbp": (50, 280),
    "dbp": (20, 200),
    "spo2|o2_saturation": (0, 100),
    "temperature_c": (30, 43),

    # labs (common)
    "cmp_lactate|lactate": (0.2, 20),
    "cmp_creatinine|creatinine": (0.1, 15),
    "cmp_bicarbonate|bicarbonate": (5, 50),
    "cmp_glucose|glucose": (20, 1000),
    "cmp_potassium|potassium": (2, 7),
    "cmp_bun|bun": (1, 200),
    "cmp_aniongap|aniongap": (0, 50),
    "cbc_wbc|wbc": (0.1, 200),
    "cbc_hemoglobin|hemoglobin": (3, 25),
    "cbc_hematocrit|hematocrit": (5, 80),
    "cbc_platelet|platelet": (5, 1500),
    "aptt_inr|inr": (0.5, 10),
    "aptt_ptt|ptt|pt": (5, 200),
    "abg_ph|ph": (6.8, 7.8),
    "base_excess": (-30, 30),
    "sodium": (110, 180),
    "chloride": (70, 130),
    "calcium": (4, 16),
    "magnesium": (0.5, 6),
    "phosphate": (0.5, 15),
    "fibrinogen": (50, 1200),
    "bilirubin_total|bilirubin": (0.1, 40),
    "alt|ast|alp": (0, 20000),
}

def col_bounds(colname):
    for k, rng in bounds.items():
        if any(tok in colname for tok in k.split("|")):
            return rng
    return None

def scan_df(df: pd.DataFrame, name: str):
    rows = []
    for c in df.columns:
        s = pd.to_numeric(df[c], errors="coerce")
        if s.notna().sum() == 0:
            continue
        lo_hi = col_bounds(c)
        n_sentinel = int(s.isin(sentinels).sum())
        n_absurd = int((s.abs() > 1e4).sum())
        n_out_phys = None
        if lo_hi:
            lo, hi = lo_hi
            n_out_phys = int(((s < lo) | (s > hi)).sum())
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lo_iqr, hi_iqr = q1 - 1.5*iqr, q3 + 1.5*iqr
        n_iqr = int(((s < lo_iqr) | (s > hi_iqr)).sum())
        rows.append({
            "feature": c,
            "n": int(s.notna().sum()),
            "std": float(s.std(skipna=True)),
            "max": float(s.max(skipna=True)),
            "n_absurd(>1e4)": n_absurd,
            "n_sentinel": n_sentinel,
            "n_out_of_physio": n_out_phys if lo_hi else None,
            "n_iqr_outliers": n_iqr,
        })
    out = pd.DataFrame(rows).sort_values(
        ["n_absurd(>1e4)", "n_sentinel", "n_out_of_physio", "n_iqr_outliers"],
        ascending=False, na_position="last"
    ).reset_index(drop=True)
    print(f"\n=== Outlier scan: {name} ===")
    display(out.head(30))
    return out

scan_sepsis = scan_df(X_sepsis, "Sepsis train_X.csv (37 features)")
scan_aki    = scan_df(X_aki,    "AKI train_X.csv (~108 features)")

# Show top offending rows for key known-problem features (lactate, RR)
def show_top(df, col_like, k=5):
    cols = [c for c in df.columns if col_like in c]
    if not cols: return
    c = cols[0]
    s = pd.to_numeric(df[c], errors='coerce')
    print(f"\nTop {k} extremes for {c}:")
    display(s.sort_values(ascending=False).head(k))

show_top(X_sepsis, "cmp_lactate")
show_top(X_sepsis, "respiratory_rate_mean")
show_top(X_aki,    "lactate_mean")
show_top(X_aki,    "respiratory_rate_mean")


=== Outlier scan: Sepsis train_X.csv (37 features) ===


,feature,n,std,max,n_absurd(>1e4),n_sentinel,n_out_of_physio,n_iqr_outliers
0,abg_ph,19092,1.225177,3.866601,0,0,19092.0,2099
1,temperature_c_min,19092,0.880590,2.808733,0,0,19092.0,1694
2,dbp_min,19092,1.080633,9.947167,0,0,19092.0,1149
3,sbp_min,19092,1.230549,8.723528,0,0,19092.0,1146
4,cmp_glucose,19092,1.279975,14.119114,0,0,19092.0,1047
5,sbp_mean,19092,1.076089,21.746982,0,0,19092.0,980
6,heart_rate_min,19092,1.253922,4.988803,0,0,19092.0,466
7,respiratory_rate_max,19092,0.786967,108.729010,0,0,19092.0,388
8,respiratory_rate_mean,19092,0.786970,108.728843,0,0,19092.0,321
9,dbp_max,19092,0.789170,107.571427,0,0,19091.0,781



=== Outlier scan: AKI train_X.csv (~108 features) ===


,feature,n,std,max,n_absurd(>1e4),n_sentinel,n_out_of_physio,n_iqr_outliers
0,ast_max,96438,1022.312426,42606.000,274,51,37.0,44025
1,ast_mean,96438,755.628708,34532.500,129,41,14.0,44915
2,ast_min,96438,561.311322,26790.000,63,56,3.0,46267
3,alt_max,96438,561.324745,24220.000,58,36,2.0,45769
4,alt_mean,96438,459.048348,20243.334,24,25,1.0,45816
5,dbp_max,96438,949.488649,114109.000,13,993,144.0,3110
6,alt_min,96438,376.986258,14770.000,13,36,0.0,45681
7,sbp_max,96438,3349.272704,1025100.000,9,170,33.0,2316
8,spo2_max,96438,4989.256581,981023.000,8,9003,62.0,6339
9,spo2_mean,96438,199.607626,39325.400,6,400,53.0,1818



Top 5 extremes for cmp_lactate:


...

8118     108.728843
7201       0.026490
725        0.007195
18092      0.003252
5143       0.002955
Name: respiratory_rate_mean, dtype: float64


Top 5 extremes for lactate_mean:


34174    425368.40000
52034        28.00000
30652        26.66000
15204        25.01111
61208        24.60000
Name: lactate_mean, dtype: float64


Top 5 extremes for respiratory_rate_mean:


590      269266.78000
44446     90617.38000
64164       719.57140
44143       184.58824
14579       124.40625
Name: respiratory_rate_mean, dtype: float64